# Repository guide: Kabyle-XLS-R Tarifit adaptation

Original code and recorded outputs retained. Read ../../docs/RUNNING.md before execution.


# Kabyle-specialized XLS-R → Tarifit — V1.2 no augmentation

This notebook starts from `Akashpb13/Kabyle_xlsr`, replaces its Kabyle CTC head with the project's exact 34-token Tarifit head, and fine-tunes it on the frozen V1.2 train/validation partitions.

**Controlled experiment policy**

- The data, tokenizer, preprocessing, optimization settings, and eight-epoch schedule match the generic XLS-R V1.2 no-augmentation run.
- The only intended model-lineage difference is the starting checkpoint: Kabyle-specialized XLS-R instead of generic XLS-R.
- Neither external waveform augmentation nor internal SpecAugment is used.
- Early stopping is disabled; the best checkpoint is selected afterward by validation CER.
- `[UNK]` and other special tokens are removed during decoding so they can never be counted as literal characters.
- Checkpoints and final weights are written to `MyDrive` and verified after saving.
- The held-out test set is not loaded or evaluated in this notebook.


In [ ]:
# Cell 1 — Install reproducible XLS-R V1.2 dependencies

!pip -q install "transformers==4.57.1" "datasets==4.4.1" "accelerate>=1.10,<2" "jiwer==4.0.0" "safetensors>=0.4.5" "soundfile>=0.12.1"


In [ ]:
# Cell 2 — Mount Google Drive and define the controlled transfer paths

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

# Read the frozen corpus and shared XLS-R preprocessing assets from the project.
PROJECT_ROOT = Path(
    "/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm"
)

# Write large checkpoints to MyDrive so their weight files are retained.
SAFE_PROJECT_ROOT = Path(
    "/content/drive/MyDrive/tarifit_asr_tfm"
)

AUGMENTATION_CONDITION = "noaug"
assert AUGMENTATION_CONDITION == "noaug"

METADATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "metadata"
    / "segments_metadata_v1_2.csv"
)

FROZEN_METADATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "metadata"
    / "segments_metadata_v1_2_train_val_frozen.csv"
)

# Reuse the exact tokenizer and preprocessed waveforms used by generic XLS-R.
TOKENIZER_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "xlsr_tokenizer_v1_2"
)

DATASET_CACHE_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "xlsr_corpus_v1_2"
)

EXPERIMENT_NAME = "kabyle_xlsr_tarifit_v1_2_noaug_full8_corrected"

MODEL_DIR = SAFE_PROJECT_ROOT / "models" / EXPERIMENT_NAME
RESULTS_DIR = SAFE_PROJECT_ROOT / "results" / EXPERIMENT_NAME

BASE_MODEL_ID = "Akashpb13/Kabyle_xlsr"
PREPROCESSING_REFERENCE_MODEL_ID = "facebook/wav2vec2-xls-r-300m"
SEED = 42

TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Source project exists:", PROJECT_ROOT.exists())
print("Safe output root exists:", SAFE_PROJECT_ROOT.exists())
print("Metadata exists:", METADATA_PATH.exists())
print("Frozen train/validation metadata exists:", FROZEN_METADATA_PATH.exists())
print("Starting checkpoint:", BASE_MODEL_ID)
print("Augmentation condition:", AUGMENTATION_CONDITION)
print("Experiment name:", EXPERIMENT_NAME)
print("Model output:", MODEL_DIR)
print("Results output:", RESULTS_DIR)


Mounted at /content/drive
Source project exists: True
Safe output root exists: True
Metadata exists: True
Frozen train/validation metadata exists: True
Starting checkpoint: Akashpb13/Kabyle_xlsr
Augmentation condition: noaug
Experiment name: kabyle_xlsr_tarifit_v1_2_noaug_full8_corrected
Model output: /content/drive/MyDrive/tarifit_asr_tfm/models/kabyle_xlsr_tarifit_v1_2_noaug_full8_corrected
Results output: /content/drive/MyDrive/tarifit_asr_tfm/results/kabyle_xlsr_tarifit_v1_2_noaug_full8_corrected


In [ ]:
# Cell 3 — Record software and GPU environment

import sys
import random
import hashlib

import torch
import transformers
import datasets
import jiwer
import soundfile
import numpy as np

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("JiWER:", jiwer.__version__ if hasattr(jiwer, "__version__") else "unknown")
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    properties = torch.cuda.get_device_properties(0)
    print("GPU memory (GB):", round(properties.total_memory / 1024**3, 2))


Python: 3.13.15
PyTorch: 2.11.0+cu128
Transformers: 4.57.1
Datasets: 4.4.1
JiWER: unknown
CUDA available: True
GPU: Tesla T4
GPU memory (GB): 14.56


In [ ]:
# Cell 4 — Load and verify the frozen V1.2 train/validation partitions

import pandas as pd

if not FROZEN_METADATA_PATH.exists():
    raise FileNotFoundError(
        "The frozen V1.2 train/validation metadata was not found. "
        "Do not train until the same frozen split used by the controlled experiments is available."
    )

frozen_df = pd.read_csv(FROZEN_METADATA_PATH)

required_columns = {
    "segment_id",
    "recording_id",
    "speaker_group_id",
    "dataset_split",
    "duration_seconds",
    "audio_path",
    "transcription",
}

missing_columns = required_columns - set(frozen_df.columns)
assert not missing_columns, f"Missing columns: {missing_columns}"

frozen_df["dataset_split"] = (
    frozen_df["dataset_split"].astype(str).str.lower().str.strip()
)
frozen_df["transcription"] = (
    frozen_df["transcription"].fillna("").astype(str).str.strip()
)

assert frozen_df["dataset_split"].isin(["train", "validation"]).all()
assert frozen_df["transcription"].ne("").all()
assert not frozen_df["segment_id"].duplicated().any()

train_df = frozen_df[frozen_df["dataset_split"] == "train"].copy()
validation_df = frozen_df[
    frozen_df["dataset_split"] == "validation"
].copy()

print("Train segments:", len(train_df))
print("Validation segments:", len(validation_df))
print(
    "Train duration:",
    round(train_df["duration_seconds"].sum() / 3600, 3),
    "hours",
)
print(
    "Validation duration:",
    round(validation_df["duration_seconds"].sum() / 3600, 3),
    "hours",
)
print("Train speakers:", sorted(train_df["speaker_group_id"].unique()))
print(
    "Validation speakers:",
    sorted(validation_df["speaker_group_id"].unique()),
)

full_metadata_df = pd.read_csv(METADATA_PATH)
full_metadata_df["dataset_split"] = (
    full_metadata_df["dataset_split"].astype(str).str.lower().str.strip()
)

train_speakers = set(train_df["speaker_group_id"].dropna())
validation_speakers = set(validation_df["speaker_group_id"].dropna())
test_speakers = set(
    full_metadata_df.loc[
        full_metadata_df["dataset_split"] == "test",
        "speaker_group_id",
    ].dropna()
)

assert not train_speakers & validation_speakers
assert not train_speakers & test_speakers
assert not validation_speakers & test_speakers

missing_audio = [
    str(PROJECT_ROOT / path)
    for path in frozen_df["audio_path"]
    if not (PROJECT_ROOT / path).exists()
]

print("Missing audio files:", len(missing_audio))
assert not missing_audio, (
    "Some frozen audio files are missing. First missing file: "
    + (missing_audio[0] if missing_audio else "")
)

metadata_sha256 = hashlib.sha256(
    FROZEN_METADATA_PATH.read_bytes()
).hexdigest()

print("Frozen metadata SHA-256:", metadata_sha256)
print("Speaker-independent partition verification completed.")


Train segments: 1754
Validation segments: 129
Train duration: 5.223 hours
Validation duration: 0.298 hours
Train speakers: ['SPK001', 'SPK002', 'SPK009']
Validation speakers: ['SPK007', 'SPK010']
Missing audio files: 0
Frozen metadata SHA-256: 4911f46e0a8c3656089677b8899d8296b9e1528d8aa3633a64728eb645f28fab
Speaker-independent partition verification completed.


In [ ]:
# Cell 5 — Verify the V1.2 orthographic character inventory

from collections import Counter
import unicodedata

FINAL_LETTERS = (
    list("abcdefghijklmnpqrstuvwxyz")
    + ["ɛ", "ɣ", "ʷ", "ḍ", "ḥ", "ṭ"]
)

ALLOWED_CHARACTERS = set(FINAL_LETTERS) | {" "}

all_text = " ".join(frozen_df["transcription"].tolist())
character_counts = Counter(all_text)

unexpected_characters = {
    character: count
    for character, count in character_counts.items()
    if character not in ALLOWED_CHARACTERS
}

print("Expected letters:", " ".join(FINAL_LETTERS))
print("Number of letters:", len(FINAL_LETTERS))
print("Unexpected characters:", unexpected_characters)

for text in frozen_df["transcription"]:
    assert text == unicodedata.normalize("NFC", text)

assert not unexpected_characters, (
    "The V1.2 transcripts contain characters outside the declared vocabulary."
)

print("V1.2 character inventory verified.")


Expected letters: a b c d e f g h i j k l m n p q r s t u v w x y z ɛ ɣ ʷ ḍ ḥ ṭ
Number of letters: 31
Unexpected characters: {}
V1.2 character inventory verified.


In [ ]:
# Cell 6 — Build and save the XLS-R V1.2 CTC tokenizer

import json
from transformers import (
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor,
)

vocab_tokens = ["|"] + FINAL_LETTERS + ["[UNK]", "[PAD]"]
vocab_dict = {
    token: index
    for index, token in enumerate(vocab_tokens)
}

VOCAB_PATH = TOKENIZER_DIR / "vocab.json"

with open(VOCAB_PATH, "w", encoding="utf-8") as file:
    json.dump(vocab_dict, file, ensure_ascii=False, indent=2)

tokenizer = Wav2Vec2CTCTokenizer(
    str(VOCAB_PATH),
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|",
    bos_token=None,
    eos_token=None,
    do_lower_case=False,
)

feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16000,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True,
)

processor = Wav2Vec2Processor(
    feature_extractor=feature_extractor,
    tokenizer=tokenizer,
)

processor.save_pretrained(TOKENIZER_DIR)

print("Vocabulary path:", VOCAB_PATH)
print("Tokenizer size:", len(tokenizer))
print("CTC blank/padding id:", tokenizer.pad_token_id)
print("Unknown-token id:", tokenizer.unk_token_id)
print("Vocabulary:", tokenizer.get_vocab())


Vocabulary path: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/data/processed/xlsr_tokenizer_v1_2/vocab.json
Tokenizer size: 34
CTC blank/padding id: 33
Unknown-token id: 32
Vocabulary: {'|': 0, 'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5, 'f': 6, 'g': 7, 'h': 8, 'i': 9, 'j': 10, 'k': 11, 'l': 12, 'm': 13, 'n': 14, 'p': 15, 'q': 16, 'r': 17, 's': 18, 't': 19, 'u': 20, 'v': 21, 'w': 22, 'x': 23, 'y': 24, 'z': 25, 'ɛ': 26, 'ɣ': 27, 'ʷ': 28, 'ḍ': 29, 'ḥ': 30, 'ṭ': 31, '[UNK]': 32, '[PAD]': 33}


In [ ]:
# Cell 7 — Verify that every V1.2 transcript is representable

unknown_segments = []

for row in frozen_df.itertuples(index=False):
    label_ids = tokenizer(row.transcription).input_ids
    if tokenizer.unk_token_id in label_ids:
        unknown_segments.append(
            (row.segment_id, row.transcription)
        )

print("Segments containing [UNK]:", len(unknown_segments))

if unknown_segments:
    for item in unknown_segments[:20]:
        print(item)

assert not unknown_segments, (
    "Some V1.2 references cannot be represented by the tokenizer."
)

print("All V1.2 references are representable.")


Segments containing [UNK]: 0
All V1.2 references are representable.


In [ ]:
# Cell 8 — Reuse or build the shared frozen V1.2 XLS-R dataset

import json
import soundfile as sf
from datasets import Dataset, DatasetDict, load_from_disk


def prepare_split(frame):
    work = frame.copy()
    work["absolute_audio_path"] = work["audio_path"].apply(
        lambda path: str(PROJECT_ROOT / path)
    )

    return Dataset.from_pandas(
        work[
            [
                "segment_id",
                "recording_id",
                "speaker_group_id",
                "duration_seconds",
                "absolute_audio_path",
                "transcription",
            ]
        ],
        preserve_index=False,
    )


def prepare_example(example):
    audio, sampling_rate = sf.read(
        example["absolute_audio_path"],
        dtype="float32",
        always_2d=False,
    )

    if sampling_rate != 16000:
        raise ValueError(
            f'{example["segment_id"]}: expected 16000 Hz, '
            f"found {sampling_rate} Hz"
        )

    if getattr(audio, "ndim", 1) != 1:
        raise ValueError(
            f'{example["segment_id"]}: audio is not mono'
        )

    model_inputs = processor(
        audio,
        sampling_rate=16000,
    )

    labels = tokenizer(
        example["transcription"]
    ).input_ids

    return {
        "input_values": model_inputs.input_values[0],
        "input_length": len(model_inputs.input_values[0]),
        "labels": labels,
    }


CACHE_MANIFEST_PATH = DATASET_CACHE_DIR / "cache_manifest.json"
cache_is_valid = False

if DATASET_CACHE_DIR.exists() and CACHE_MANIFEST_PATH.exists():
    with open(CACHE_MANIFEST_PATH, "r", encoding="utf-8") as file:
        cache_manifest = json.load(file)

    cached_preprocessing_model = cache_manifest.get(
        "preprocessing_reference_model",
        cache_manifest.get("base_model"),
    )

    cache_is_valid = (
        cache_manifest.get("metadata_sha256") == metadata_sha256
        and cache_manifest.get("tokenizer_size") == len(tokenizer)
        and cached_preprocessing_model == PREPROCESSING_REFERENCE_MODEL_ID
    )

if DATASET_CACHE_DIR.exists() and not cache_is_valid:
    raise RuntimeError(
        "The shared XLS-R V1.2 cache does not match the frozen metadata "
        "or tokenizer. Inspect it before training."
    )

if cache_is_valid:
    xlsr_dataset = load_from_disk(str(DATASET_CACHE_DIR))
    print("Reused the validated generic-XLS-R V1.2 preprocessing cache.")
else:
    raw_dataset = DatasetDict(
        {
            "train": prepare_split(train_df),
            "validation": prepare_split(validation_df),
        }
    )

    xlsr_dataset = raw_dataset.map(
        prepare_example,
        remove_columns=[
            "absolute_audio_path",
            "transcription",
            "recording_id",
            "speaker_group_id",
            "duration_seconds",
        ],
        num_proc=1,
        desc="Preparing V1.2 16 kHz audio and CTC labels",
    )

    DATASET_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    xlsr_dataset.save_to_disk(str(DATASET_CACHE_DIR))

    with open(CACHE_MANIFEST_PATH, "w", encoding="utf-8") as file:
        json.dump(
            {
                "metadata_sha256": metadata_sha256,
                "tokenizer_size": len(tokenizer),
                "base_model": PREPROCESSING_REFERENCE_MODEL_ID,
                "preprocessing_reference_model": PREPROCESSING_REFERENCE_MODEL_ID,
            },
            file,
            ensure_ascii=False,
            indent=2,
        )

    print("Built and saved the shared V1.2 XLS-R preprocessing cache.")

train_ds = xlsr_dataset["train"]
validation_ds = xlsr_dataset["validation"]

assert len(train_ds) == 1754, len(train_ds)
assert len(validation_ds) == 129, len(validation_ds)

print("Train examples:", len(train_ds))
print("Validation examples:", len(validation_ds))
print("Dataset columns:", train_ds.column_names)


Reused the validated generic-XLS-R V1.2 preprocessing cache.
Train examples: 1754
Validation examples: 129
Dataset columns: ['segment_id', 'input_values', 'input_length', 'labels']


In [ ]:
# Cell 9 — Summarize the processed V1.2 audio durations

def duration_summary(dataset_split):
    seconds = np.asarray(
        dataset_split["input_length"],
        dtype=np.float64,
    ) / 16000.0

    return {
        "segments": len(seconds),
        "hours": float(seconds.sum() / 3600),
        "minimum_seconds": float(seconds.min()),
        "maximum_seconds": float(seconds.max()),
        "mean_seconds": float(seconds.mean()),
    }


print("Train:", duration_summary(train_ds))
print("Validation:", duration_summary(validation_ds))


Train: {'segments': 1754, 'hours': 5.222733593749999, 'minimum_seconds': 0.848, 'maximum_seconds': 19.984, 'mean_seconds': 10.71940760404789}
Validation: {'segments': 129, 'hours': 0.2983577777777778, 'minimum_seconds': 0.944, 'maximum_seconds': 26.0, 'mean_seconds': 8.326263565891473}


In [ ]:
# Cell 10 — Load Kabyle-specialized XLS-R with a fresh Tarifit CTC head

from transformers import Wav2Vec2ForCTC

APPLY_SPEC_AUGMENT = False
MASK_TIME_PROB = 0.0
MASK_TIME_LENGTH = 5

model = Wav2Vec2ForCTC.from_pretrained(
    BASE_MODEL_ID,
    vocab_size=len(tokenizer),
    pad_token_id=tokenizer.pad_token_id,
    ctc_loss_reduction="mean",
    ctc_zero_infinity=True,
    attention_dropout=0.0,
    hidden_dropout=0.0,
    feat_proj_dropout=0.0,
    apply_spec_augment=APPLY_SPEC_AUGMENT,
    mask_time_prob=MASK_TIME_PROB,
    mask_time_length=MASK_TIME_LENGTH,
    mask_feature_prob=0.0,
    layerdrop=0.0,
    ignore_mismatched_sizes=True,
)

# Preserve the Kabyle-trained Transformer layers while replacing the
# incompatible Kabyle output head with the new 34-token Tarifit head.
model.freeze_feature_encoder()

assert model.config.vocab_size == len(tokenizer) == 34
assert model.config.pad_token_id == tokenizer.pad_token_id
assert model.lm_head.out_features == len(tokenizer)
assert model.config.apply_spec_augment is False

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)
trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print("Starting checkpoint:", BASE_MODEL_ID)
print("Model vocabulary:", model.config.vocab_size)
print("Tokenizer vocabulary:", len(tokenizer))
print("CTC head:", model.lm_head)
print("Total parameters:", f"{total_parameters:,}")
print("Trainable parameters:", f"{trainable_parameters:,}")
print("External waveform augmentation: none")
print("Internal SpecAugment enabled:", model.config.apply_spec_augment)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

Some weights of the model checkpoint at Akashpb13/Kabyle_xlsr were not used when initializing Wav2Vec2ForCTC: ['wav2vec2.masked_spec_embed']
- This IS expected if you are initializing Wav2Vec2ForCTC from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing Wav2Vec2ForCTC from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at Akashpb13/Kabyle_xlsr and are newly initialized because the shapes did not match:
- lm_head.bias: found shape torch.Size([57]) in the checkpoint and torch.Size([34]) in the model instantiated
- lm_head.weight: found shape torch.Size([57, 1024]) in the checkpoint and torch.Size([34, 1024]) in the model instantiated

Starting checkpoint: Akashpb13/Kabyle_xlsr
Model vocabulary: 34
Tokenizer vocabulary: 34
CTC head: Linear(in_features=1024, out_features=34, bias=True)
Total parameters: 315,472,546
Trainable parameters: 311,262,370
External waveform augmentation: none
Internal SpecAugment enabled: False


In [ ]:
# Cell 11 — Check CTC feasibility of every V1.2 example

def minimum_ctc_frames(labels):
    repeated_adjacent_labels = sum(
        labels[index] == labels[index - 1]
        for index in range(1, len(labels))
    )
    return len(labels) + repeated_adjacent_labels


def find_ctc_infeasible(dataset_split):
    invalid = []

    for index, example in enumerate(dataset_split):
        input_samples = int(example["input_length"])
        output_frames = int(
            model._get_feat_extract_output_lengths(
                torch.tensor(input_samples)
            ).item()
        )

        labels = example["labels"]
        required_frames = minimum_ctc_frames(labels)

        if output_frames < required_frames:
            invalid.append(
                {
                    "index": index,
                    "segment_id": example.get("segment_id", index),
                    "audio_seconds": input_samples / 16000.0,
                    "output_frames": output_frames,
                    "label_length": len(labels),
                    "minimum_ctc_frames": required_frames,
                }
            )

    return invalid


bad_train = find_ctc_infeasible(train_ds)
bad_validation = find_ctc_infeasible(validation_ds)

print("CTC-infeasible training examples:", len(bad_train))
print("CTC-infeasible validation examples:", len(bad_validation))

if bad_train:
    display(pd.DataFrame(bad_train))
if bad_validation:
    display(pd.DataFrame(bad_validation))

assert not bad_train, (
    "Training contains CTC-infeasible examples. Inspect before training."
)
assert not bad_validation, (
    "Validation contains CTC-infeasible examples. Inspect before training."
)

print("All V1.2 examples are CTC-feasible.")


CTC-infeasible training examples: 0
CTC-infeasible validation examples: 0
All V1.2 examples are CTC-feasible.


In [ ]:
# Cell 12 — Define dynamic CTC padding for audio and labels

from dataclasses import dataclass
from typing import Dict, List, Union


@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(
        self,
        features: List[Dict[str, Union[List[int], torch.Tensor]]],
    ) -> Dict[str, torch.Tensor]:
        input_features = [
            {"input_values": feature["input_values"]}
            for feature in features
        ]

        label_features = [
            {"input_ids": feature["labels"]}
            for feature in features
        ]

        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )

        labels_batch = self.processor.tokenizer.pad(
            label_features,
            padding=self.padding,
            return_tensors="pt",
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch["attention_mask"].ne(1),
            -100,
        )

        batch["labels"] = labels
        return batch


data_collator = DataCollatorCTCWithPadding(
    processor=processor,
    padding=True,
)

print("CTC data collator ready.")


CTC data collator ready.


In [ ]:
# Cell 13 — Define corrected validation WER and CER metrics

from jiwer import wer, cer


def decode_predictions(prediction_ids):
    return tokenizer.batch_decode(
        prediction_ids,
        skip_special_tokens=True,
    )


def decode_references(label_ids):
    return tokenizer.batch_decode(
        label_ids,
        group_tokens=False,
        skip_special_tokens=True,
    )


def compute_metrics(prediction):
    prediction_ids = np.argmax(
        prediction.predictions,
        axis=-1,
    )

    label_ids = prediction.label_ids.copy()
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    prediction_text = decode_predictions(prediction_ids)
    reference_text = decode_references(label_ids)

    assert not any("[UNK]" in text for text in prediction_text)
    assert not any("[UNK]" in text for text in reference_text)

    return {
        "wer": wer(reference_text, prediction_text) * 100,
        "cer": cer(reference_text, prediction_text) * 100,
    }


# Confirm that [UNK] is treated as a special symbol and never as five letters.
decode_probe = tokenizer.decode(
    [tokenizer.unk_token_id],
    skip_special_tokens=True,
)
assert decode_probe == "", repr(decode_probe)

print("Corrected WER/CER metrics ready.")
print("Special tokens removed during decoding: yes")
print("[UNK] literal-character inflation prevented: yes")


Corrected WER/CER metrics ready.
Special tokens removed during decoding: yes
[UNK] literal-character inflation prevented: yes


In [ ]:
# Cell 14 — Save the controlled Kabyle-to-Tarifit configuration

experiment_configuration = {
    "experiment": EXPERIMENT_NAME,
    "experimental_role": "related-language transfer from Kabyle to Tarifit",
    "base_model": BASE_MODEL_ID,
    "comparison_model": PREPROCESSING_REFERENCE_MODEL_ID,
    "augmentation_condition": AUGMENTATION_CONDITION,
    "metadata_path": str(FROZEN_METADATA_PATH),
    "metadata_sha256": metadata_sha256,
    "tokenizer_path": str(TOKENIZER_DIR),
    "tokenizer_size": len(tokenizer),
    "final_letters": FINAL_LETTERS,
    "seed": SEED,
    "external_waveform_augmentation": False,
    "internal_spec_augment": APPLY_SPEC_AUGMENT,
    "internal_mask_time_prob": MASK_TIME_PROB,
    "internal_mask_time_length": MASK_TIME_LENGTH,
    "feature_encoder_frozen": True,
    "planned_epochs": 8,
    "early_stopping": False,
    "per_device_train_batch_size": 2,
    "per_device_eval_batch_size": 2,
    "gradient_accumulation_steps": 8,
    "effective_train_batch_size": 16,
    "learning_rate": 3e-5,
    "weight_decay": 0.01,
    "warmup_steps": 100,
    "fp16": bool(torch.cuda.is_available()),
    "gradient_checkpointing": True,
    "checkpoint_selection_metric": "validation CER",
    "special_tokens_removed_during_decoding": True,
    "held_out_test_used": False,
}

CONFIGURATION_PATH = RESULTS_DIR / "experiment_configuration.json"

with open(CONFIGURATION_PATH, "w", encoding="utf-8") as file:
    json.dump(
        experiment_configuration,
        file,
        ensure_ascii=False,
        indent=2,
    )

print("Saved configuration:", CONFIGURATION_PATH)


Saved configuration: /content/drive/MyDrive/tarifit_asr_tfm/results/kabyle_xlsr_tarifit_v1_2_noaug_full8_corrected/experiment_configuration.json


In [ ]:
# Cell 15 — Configure the matched eight-epoch Trainer

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=str(MODEL_DIR),
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    group_by_length=True,
    length_column_name="input_length",
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_steps=100,
    lr_scheduler_type="linear",
    num_train_epochs=8,
    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="cer",
    greater_is_better=False,
    logging_strategy="steps",
    logging_steps=25,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
    dataloader_num_workers=2,
    remove_unused_columns=True,
    save_safetensors=True,
)

assert training_args.num_train_epochs == 8
assert training_args.metric_for_best_model == "cer"
assert training_args.load_best_model_at_end is True

print(training_args)


TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=42,
dataloader_drop_last=False,
dataloader_num_workers=2,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
f

In [ ]:
# Cell 16 — Create the Trainer and verify every saved checkpoint

from transformers import Trainer, TrainerCallback


class CheckpointWeightVerifier(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        checkpoint_dir = (
            Path(args.output_dir)
            / f"checkpoint-{state.global_step}"
        )
        weight_files = model_weight_files(checkpoint_dir)

        if not weight_files:
            raise RuntimeError(
                f"Checkpoint has no model weights: {checkpoint_dir}"
            )

        print(
            "Verified checkpoint weights:",
            checkpoint_dir,
            [path.name for path in weight_files],
        )
        return control


def model_weight_files(directory):
    directory = Path(directory)
    candidates = [
        directory / "model.safetensors",
        directory / "pytorch_model.bin",
        directory / "model.safetensors.index.json",
        directory / "pytorch_model.bin.index.json",
    ]
    candidates.extend(directory.glob("model-*.safetensors"))
    candidates.extend(directory.glob("pytorch_model-*.bin"))
    return sorted({path for path in candidates if path.exists()})


trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    train_dataset=train_ds,
    eval_dataset=validation_ds,
    processing_class=processor,
    callbacks=[CheckpointWeightVerifier()],
)

print("Trainer ready.")
print("Early stopping: disabled")
print("Train examples:", len(train_ds))
print("Validation examples:", len(validation_ds))
print("Planned epochs:", training_args.num_train_epochs)
print("Checkpoint selection: lowest validation CER")


Trainer ready.
Early stopping: disabled
Train examples: 1754
Validation examples: 129
Planned epochs: 8
Checkpoint selection: lowest validation CER


In [ ]:
# Cell 17 — Start or safely resume Kabyle-to-Tarifit fine-tuning

from transformers.trainer_utils import get_last_checkpoint

last_checkpoint = get_last_checkpoint(str(MODEL_DIR))

if last_checkpoint is not None:
    last_checkpoint = Path(last_checkpoint)
    checkpoint_weights = model_weight_files(last_checkpoint)
    assert checkpoint_weights, (
        f"Cannot resume because weights are missing: {last_checkpoint}"
    )
    print("Resuming from verified checkpoint:", last_checkpoint)
    resume_path = str(last_checkpoint)
else:
    print("Starting a fresh controlled V1.2 run.")
    resume_path = None

train_result = trainer.train(
    resume_from_checkpoint=resume_path,
)

print("Training completed.")
print(train_result)
print("Completed global steps:", trainer.state.global_step)
print("Completed epoch:", trainer.state.epoch)
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best validation CER:", trainer.state.best_metric)

assert trainer.state.global_step > 0
assert trainer.state.best_model_checkpoint is not None


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None}.


Starting a fresh controlled V1.2 run.


/usr/local/lib/python3.13/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


Epoch,Training Loss,Validation Loss,Wer,Cer
1,3.002200,3.284545,100.000000,100.000000
2,2.913100,3.265694,100.000000,100.000000
3,1.811800,3.117551,100.467687,67.175818
4,1.091800,3.130781,100.467687,53.605595
5,0.854000,3.445789,102.168367,53.075006
6,0.790400,3.249627,100.935374,52.488142
7,0.673400,3.490165,102.168367,52.882064
8,0.696500,3.389185,100.722789,52.279122


Verified checkpoint weights: /content/drive/MyDrive/tarifit_asr_tfm/models/kabyle_xlsr_tarifit_v1_2_noaug_full8_corrected/checkpoint-110 ['model.safetensors']


...

Verified checkpoint weights: /content/drive/MyDrive/tarifit_asr_tfm/models/kabyle_xlsr_tarifit_v1_2_noaug_full8_corrected/checkpoint-660 ['model.safetensors']


/usr/local/lib/python3.13/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


Verified checkpoint weights: /content/drive/MyDrive/tarifit_asr_tfm/models/kabyle_xlsr_tarifit_v1_2_noaug_full8_corrected/checkpoint-770 ['model.safetensors']


/usr/local/lib/python3.13/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


Verified checkpoint weights: /content/drive/MyDrive/tarifit_asr_tfm/models/kabyle_xlsr_tarifit_v1_2_noaug_full8_corrected/checkpoint-880 ['model.safetensors']
Training completed.
TrainOutput(global_step=880, training_loss=1.9803258378397335, metrics={'train_runtime': 2946.9637, 'train_samples_per_second': 4.762, 'train_steps_per_second': 0.299, 'total_flos': 4.566639967258468e+18, 'train_loss': 1.9803258378397335, 'epoch': 8.0})
Completed global steps: 880
Completed epoch: 8.0
Best checkpoint: /content/drive/MyDrive/tarifit_asr_tfm/models/kabyle_xlsr_tarifit_v1_2_noaug_full8_corrected/checkpoint-880
Best validation CER: 52.279122115925716


In [ ]:
# Cell 18 — Evaluate, save, and verify the selected best model

best_validation_metrics = trainer.evaluate(
    eval_dataset=validation_ds,
    metric_key_prefix="validation",
)

print("Best-checkpoint validation metrics:")
for key, value in best_validation_metrics.items():
    print(f"{key}: {value}")

BEST_MODEL_DIR = MODEL_DIR / "best_model"
trainer.save_model(str(BEST_MODEL_DIR))
processor.save_pretrained(BEST_MODEL_DIR)

best_weight_files = model_weight_files(BEST_MODEL_DIR)

print("Saved best model:", BEST_MODEL_DIR)
print("Verified final weight files:", [path.name for path in best_weight_files])

assert best_weight_files, "The saved best model has no weight file."


Best-checkpoint validation metrics:
validation_loss: 3.3891913890838623
validation_wer: 100.72278911564625
validation_cer: 52.279122115925716
validation_runtime: 10.9901
validation_samples_per_second: 11.738
validation_steps_per_second: 5.914
epoch: 8.0
Saved best model: /content/drive/MyDrive/tarifit_asr_tfm/models/kabyle_xlsr_tarifit_v1_2_noaug_full8_corrected/best_model
Verified final weight files: ['model.safetensors']


In [ ]:
# Cell 19 — Save corrected V1.2 validation predictions

prediction_output = trainer.predict(
    validation_ds,
    metric_key_prefix="validation_prediction",
)

prediction_ids = np.argmax(
    prediction_output.predictions,
    axis=-1,
)

prediction_texts = decode_predictions(prediction_ids)
reference_texts = [
    tokenizer.decode(
        example["labels"],
        group_tokens=False,
        skip_special_tokens=True,
    )
    for example in validation_ds
]

assert not any("[UNK]" in text for text in prediction_texts)
assert not any("[UNK]" in text for text in reference_texts)

validation_predictions = pd.DataFrame(
    {
        "segment_id": validation_ds["segment_id"],
        "reference": reference_texts,
        "prediction": prediction_texts,
    }
)

validation_predictions["segment_wer"] = [
    wer(reference, prediction)
    for reference, prediction in zip(
        reference_texts,
        prediction_texts,
    )
]

validation_predictions["segment_cer"] = [
    cer(reference, prediction)
    for reference, prediction in zip(
        reference_texts,
        prediction_texts,
    )
]

PREDICTIONS_PATH = RESULTS_DIR / "validation_predictions.csv"
validation_predictions.to_csv(
    PREDICTIONS_PATH,
    index=False,
    encoding="utf-8",
)

display(validation_predictions.head(20))
print("Saved corrected predictions:", PREDICTIONS_PATH)


,segment_id,reference,prediction,segment_wer,segment_cer
0,REC090_SEG0010,ssalamuɛlikum necc meryem,trastɛijejen nḥayacwyteqsaḥ muhim g igas teɛuq...,18.333333,13.640000
1,REC090_SEG0011,aqay ruxxa tnayn uɛecrin sana ḍi hulanda,ijen wcma iraɣar iakidse lɛalaqamliḥ macawa xa...,2.714286,2.575000
2,REC090_SEG0012,mercex ak nmis n jjiran usiɣ ḍ zi lmeɣrib umi ...,ḥemdulia ɣar rebɛa n tawaynuɣari tnayn tiḥenej...,0.960000,0.817391
3,REC090_SEG0013,umi wsiɣd dda ufix manayenni wa dji ca min ira...,manaya maucat may tux tegḍe n ḥar yekmel mkiḍ ...,1.642857,1.484375
4,REC090_SEG0014,a necc mammec ira djjix ḍi lmeɣrib wadji manay...,waxameni d reɛqenilu i raqaɣasmaɛliceqamanaya ...,1.545455,1.716667
5,REC090_SEG0015,necc ḍi lmeɣrib ira ɣari lḥurriya inu ira ɣari...,maymirazux rebdatawin wad qimena ad qim watege...,1.000000,0.750000
6,REC090_SEG0016,ḍi lmeɣrib neccin mammec ira niɛicc,wcitcnam leflus tuɣ tsarafenɣem yɛn imem lala ...,3.666667,3.000000
7,REC090_SEG0017,ak baba dd yemma wa ɣaneɣ ca ɣaneɣ ca n reḥway...,safi iga mana yeni ḍ walidin n is wadjin tayra...,0.972222,0.729412
8,REC090_SEG0018,lmuhim wsiɣd,tɣar in zaysimkem wa zu nca tiri bla iwazu ca ...,14.000000,12.000000
9,REC090_SEG0019,necc ira ɛemmas wa wsiɣd ɣar urupa wsiɣd ḍi ṭi...,amryamaqa semliḥ aqa sm iḥmaymira rabdad eḥ c ...,1.000000,0.739726


Saved corrected predictions: /content/drive/MyDrive/tarifit_asr_tfm/results/kabyle_xlsr_tarifit_v1_2_noaug_full8_corrected/validation_predictions.csv


In [ ]:
# Cell 20 — Save history and the final controlled-run summary

history_df = pd.DataFrame(trainer.state.log_history)
HISTORY_PATH = RESULTS_DIR / "training_history.csv"
history_df.to_csv(HISTORY_PATH, index=False)

selected_checkpoint = Path(trainer.state.best_model_checkpoint)
selected_checkpoint_weights = model_weight_files(selected_checkpoint)

assert selected_checkpoint_weights, (
    "The selected checkpoint does not contain model weights."
)

summary = {
    **experiment_configuration,
    "train_segments": len(train_ds),
    "validation_segments": len(validation_ds),
    "train_duration": duration_summary(train_ds),
    "validation_duration": duration_summary(validation_ds),
    "total_parameters": int(total_parameters),
    "trainable_parameters": int(trainable_parameters),
    "completed_global_steps": int(trainer.state.global_step),
    "completed_epoch": float(trainer.state.epoch),
    "best_checkpoint": trainer.state.best_model_checkpoint,
    "best_checkpoint_weight_files": [
        path.name for path in selected_checkpoint_weights
    ],
    "best_validation_cer": (
        float(trainer.state.best_metric)
        if trainer.state.best_metric is not None
        else None
    ),
    "final_validation_metrics": {
        key: float(value)
        if isinstance(value, (int, float, np.floating))
        else value
        for key, value in best_validation_metrics.items()
    },
    "results": {
        "training_history": str(HISTORY_PATH),
        "validation_predictions": str(PREDICTIONS_PATH),
        "best_model": str(BEST_MODEL_DIR),
    },
}

SUMMARY_PATH = RESULTS_DIR / "experiment_summary.json"
with open(SUMMARY_PATH, "w", encoding="utf-8") as file:
    json.dump(summary, file, ensure_ascii=False, indent=2)

print("Saved training history:", HISTORY_PATH)
print("Saved experiment summary:", SUMMARY_PATH)
print("Verified selected checkpoint weights:", [
    path.name for path in selected_checkpoint_weights
])
print("Held-out test evaluated: no")


Saved training history: /content/drive/MyDrive/tarifit_asr_tfm/results/kabyle_xlsr_tarifit_v1_2_noaug_full8_corrected/training_history.csv
Saved experiment summary: /content/drive/MyDrive/tarifit_asr_tfm/results/kabyle_xlsr_tarifit_v1_2_noaug_full8_corrected/experiment_summary.json
Verified selected checkpoint weights: ['model.safetensors']
Held-out test evaluated: no
